# Fase 2 - Validación y depuración de datos

## Objetivo
Comprobar la coherencia temporal, revisar nulos y duplicados, validar el esfuerzo calculado y definir el set de variables X antes de pasar al modelado.

## Insumos
- `data/integrado/maestro_joven_uk.parquet`
- `data/limpia/hipotecas_limpio.parquet`
- `data/integrado/hipoteca_2022.parquet`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from proyecto import config
from proyecto import indicadores
import pandas as pd

maestro = pd.read_parquet(config.RUTA_MAESTRO).copy()
hipotecas = pd.read_parquet(config.RUTA_HIPOTECAS_LIMPIO).copy()
hipoteca_2022 = pd.read_parquet(config.RUTA_HIPOTECA_2022).copy()

print("=== 1. CONSISTENCIA TEMPORAL ===")
print(f"Maestro: {maestro.shape}")
print("Años por región:")
print(maestro.groupby('region')['anio'].nunique().to_string())
print(f"Rango: {maestro['anio'].min()} - {maestro['anio'].max()}")
print(f"Clave (anio, region) única: {maestro.duplicated(['anio','region']).sum() == 0}")

## 2. Nulos y duplicados

In [ ]:
tasa = float(hipotecas['tasa_inicial_pct'].mean())
comisiones = float(hipotecas['comisiones_total'].mean())
df = maestro.copy()
df['cuota_mensual'] = df.apply(
    lambda r: indicadores.cuota_mensual_pmt(r['precio_mediana'], tasa, config.PLAZO_ANIOS, config.LTV, comisiones),
    axis=1,
)
df['salario_joven_mensual'] = df['salario_joven_region'] / 12.0
df['esfuerzo'] = df['cuota_mensual'] / df['salario_joven_mensual']

print("\n=== 2. NULOS Y DUPLICADOS ===")
print(f"Nulos totales: {int(df.isna().sum().sum())}")
print(f"Duplicados (anio,region): {int(df.duplicated(['anio','region']).sum())}")

print("\nNulos por columna:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

## 3. Coherencia del esfuerzo

In [ ]:
print("\n=== 3. COHERENCIA DEL ESFUERZO ===")
print(df['esfuerzo'].describe())
print(f"\nFilas con esfuerzo > 0.30: {int((df['esfuerzo']>0.30).sum())} / {len(df)}")
print("\nEsfuerzo medio por región:")
print(df.groupby('region')['esfuerzo'].mean().sort_values(ascending=False).round(3).to_string())

## 4. Cruce con hipoteca_2022 y outliers tratados

In [ ]:
print("\n=== 4. CRUCE CON OUTLIERS TRATADOS ===")
print(f"Tasa media hipotecas limpia: {tasa:.3f}%")
print(f"Comisiones media: {comisiones:.2f}")
print(f"hipoteca_2022 filas: {len(hipoteca_2022)}")
print("\nhipoteca_2022 cuota_pct_salario:")
print(hipoteca_2022[['region','cuota_pct_salario','accesible']].sort_values('cuota_pct_salario').to_string(index=False))

## 5. Regla IQR en esfuerzo

In [ ]:
def regla_iqr(s):
    s_clean = s.dropna()
    if len(s_clean) == 0:
        return {}
    q1 = s_clean.quantile(0.25)
    q3 = s_clean.quantile(0.75)
    iqr = q3 - q1
    lim_sup = q3 + 1.5 * iqr
    lim_inf = q1 - 1.5 * iqr
    return {
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lim_inf": float(lim_inf),
        "lim_sup": float(lim_sup),
        "out_sup": int((s_clean > lim_sup).sum()),
        "out_inf": int((s_clean < lim_inf).sum()),
    }

print("\n=== 5. REGLA IQR EN ESFUERZO ===")
iq = regla_iqr(df['esfuerzo'])
print(iq)

## 6. Set de variables X propuesto

In [ ]:
print("\n=== 6. SET DE VARIABLES PARA X (propuesto) ===")
num_vars = ["precio_mediana", "salario_joven_region", "anos_salario", "precio_var_pct",
            "tmax_anom", "tmin_anom", "lluvia_anom", "sol_anom", "af_anom"]
print("Numéricas:", num_vars)
print("Categórica: region (convertir con get_dummies drop_first=True)")
print("Identificadores a excluir: ['anio']")
print(f"Nulos en variables propuestas: {int(df[num_vars + ['region']].isna().sum().sum())}")
print("\nImputación sugerida (igual que modelado.py):")
print("  df['sol_anom'] = df['sol_anom'].fillna(0.0)")
print("  df['precio_var_pct'] = df['precio_var_pct'].fillna(0.0)")

## Conclusión Fase 2
- Consistencia temporal OK: 10 regiones × 6 años (2015-2020), sin duplicados.
- 28 nulos totales: `sol_anom` (18) y `precio_var_pct` (10). Se imputan a 0.0.
- Esfuerzo sin valores <= 0, sin atípicos por IQR.
- **DATOS VALIDADOS OK** tras aplicar la imputación.